# B v7 — replay konflik handoff dari cache beku

Notebook eksperimen Phase 9 ini menjalankan ulang counting pilot **TGDI-WEEKDAY-SIANG / 9.128 frame sumber** dengan identitas input B v6 yang sama. Detector, tracker, ReID, dan enhancement tidak dijalankan. Konfigurasi B v6 tetap terikat hash.

Hasil B v7 adalah keputusan mesin riset; kemiripan trajectory hanya menjadi konflik identitas yang belum terverifikasi. Prediksi tidak menjadi crossing terukur. QC independen dan gate P0 harus dibaca terpisah dari keberhasilan eksekusi notebook. Baseline lama tidak ditimpa.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "AGENTS.md").is_file() and (REPO_ROOT.parent / "AGENTS.md").is_file():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "AGENTS.md").is_file(), "Set REPO_ROOT ke checkout proyek yang benar"
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# Lokasi sumber aktual di workspace eksekusi; ubah ke mount Kaggle yang memuat
# donor/cache yang SAMA dan video asli. Identitas/hash diperiksa sebelum replay.
SOURCE_RUN_ROOT = Path("/tmp/qc-donor/phase_8_tgdi_research/20261015T083731_664817Z")
SOURCE_VIDEO_ROOT = Path("/tmp/qc-source")
OUTPUT_ROOT = REPO_ROOT / "results/b_v7/execution"
RUN_ID = None  # None membuat nama baru bertimestamp; run lama tidak ditimpa.
RUN_REPLAY = False  # Ubah True setelah verifikasi dan regresi berhasil.


In [ ]:
import phase_8_b_v7_batch as batch

policy_path = REPO_ROOT / batch.POLICY_PATH
expected_policy = batch.build_policy(REPO_ROOT)
if policy_path.exists():
    assert batch._read(policy_path) == expected_policy, "Policy tersimpan berbeda; jangan ditimpa diam-diam"
else:
    batch._write(policy_path, expected_policy)

prepared = batch.prepare_b_v7_batch(
    repo_root=REPO_ROOT,
    source_run_root=SOURCE_RUN_ROOT,
    source_video_root=SOURCE_VIDEO_ROOT,
    policy_path=policy_path,
)
print({"policy_sha256": prepared["policy_sha256"],
       "baseline_result_sha256": prepared["baseline_result_sha256"],
       "source_identity": prepared["jobs"][0]["b_v2_source_identity"],
       "frames": prepared["jobs"][0]["metadata"]["frame_count"]})


In [ ]:
import subprocess

for script in ("phase_8_b_v7_handoff_tests.py", "phase_8_b_v7_counting_tests.py", "phase_8_b_v7_batch_tests.py"):
    subprocess.run([sys.executable, str(REPO_ROOT / "notebooks" / script)],
                   cwd=REPO_ROOT, check=True)


In [ ]:
if RUN_REPLAY:
    report = batch.run_b_v7_batch(
        repo_root=REPO_ROOT,
        source_run_root=SOURCE_RUN_ROOT,
        source_video_root=SOURCE_VIDEO_ROOT,
        policy_path=policy_path,
        output_root=OUTPUT_ROOT,
        run_id=RUN_ID,
    )
    print({key: report[key] for key in ("status", "frames_processed", "output_directory",
                                       "inputs_and_sources_unchanged", "wall_seconds")})
else:
    print("Verifikasi selesai; RUN_REPLAY=False sehingga replay belum dijalankan oleh cell ini.")


Output run baru berada di `results/b_v7/execution/<run_id>/`. Buka `batch_report.json`, `policy.json`, `source_snapshot.json`, `retention_manifest.json`, serta `weekday_siang/counting_result.json.gz`, `v7_handoff_audit.json.gz`, dan `v7_publication_audit.json.gz`. Keberadaan output tidak berarti P0 lulus; bandingkan event dengan B v6 dan tinjau sumber temporal secara independen. Jangan menjumlahkan accepted, pending, dan invalid sebagai jumlah kendaraan fisik.
